# Ingesta de datos en streaming con arquitectura medallion

## Indicaciones generales

### Errores en la ejecución 

Si tras una parada obtienes errores tipo:
- DELTA_INVALID_SOURCE_VERSION 
- DELTA_FILE_NOT_FOUND_DT

Deberías borrar los checkpoints para evitar inconsistencias, especialmente en la capa gold.

### Problemas de falta de recursos

Dado que quieres lanzar procesos en batch, además de streaming, es recomendable reservar un  core para batch y los otros tres para streaming:

spark-submit \
  --conf spark.cores.max=3 \
  --conf spark.sql.shuffle.partitions=6 \
  streaming_job
  
### Creación de sesión

**.config("spark.streaming.stopGracefullyOnShutdown", "true")**  si el proceso de streaming se detiene (por ejemplo, Ctrl+C, kill, parada del contenedor, etc), es recomendable hacer una parada controlada. Para ello se recomienda incluir esta opción en el inicio de sesión para:
- Terminar de procesar los micro‑batches que ya estén en curso.
- Hace flush de los sinks (escrituras pendientes)  y
- Actualizar checkpoints/offsets antes de parar.

Es decir, intenta que el estado final sea ncoherente: menos riesgo de perder datos a medias o de duplicar mucho al reiniciar.

In [1]:
from pyspark.sql import SparkSession
from delta import *
spark = (SparkSession.builder 
    .appName("Kafka2Spark4-Medallion") 
    .master("spark://spark-master:7077") 
    .config("spark.submit.deployMode", "client") 
    .config("spark.ui.enabled", "true").config("spark.ui.port", "4040").config("spark.driver.bindAddress", "0.0.0.0")          
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") 
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123") 
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") 
    .config("spark.hadoop.fs.s3a.path.style.access", "true") 
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") 
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")  
    .config("spark.jars.packages","org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.1") 
    .config("spark.jars","/opt/spark/jars/hadoop-aws-3.4.0.jar, /opt/spark/jars/aws-java-sdk-bundle-2.23.19.jar, /opt/spark/jars/delta-spark_2.13-4.0.0.jar, /opt/spark/jars/delta-storage-4.0.0.jar, /opt/spark/jars/antlr4-runtime-4.13.1.jar, , /opt/spark/jars/spark-sql-kafka-0-10_2.13-4.0.1.jar, /opt/spark/jars/kafka-clients-3.9.1.jar") 
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") 
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.streaming.stopGracefullyOnShutdown", "true")
    .getOrCreate()
    )
# ajuste para latencia estable
spark.conf.set("spark.streaming.backpressure.enabled", "true")


print("SparkSession creada")

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jovyan/.ivy2.5.2/cache
The jars for the packages stored in: /home/jovyan/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b155a3e1-2b02-4b39-86e4-b4c029ad4da1;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.0.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.0.1 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found org.slf4j#slf4j-api;2.0.16 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.1 in central
	found org.apache.hadoop#hadoop-client-api;3.4.1 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala

SparkSession creada


## Creación de tablas delta para bronze, silver y gold
Al tener varios streams encadenadenados (Bronze-> Silver-> Gold) las tablas delta **sink** de un stream que sean fuente para otro stream, han de ser creadas previamente. Ejemplo: /bronze/ECG_RPM es sink de bronze y fuente para silver, igual ocurre para la tabla /silver/ECG_RPM es sink de la capa silver y fuente de gold.

In [2]:
from pyspark.sql.functions import from_json, col
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StringType, TimestampType, IntegerType, FloatType, LongType, StructField

schema_bronze = StructType([
    StructField("raw_value", StringType(), True),
    StructField("topic", StringType(), True),
    StructField("offset", LongType(), True),
    StructField("ingest_time", TimestampType(), True)
])

bronze_table="s3a://hospital/bronze/ECG_RPM"

if not DeltaTable.isDeltaTable(spark, bronze_table):
    bronze_df = spark.createDataFrame([], schema_bronze)    
    # Guardar como tabla Delta en S3 
    bronze_df.write.format("delta").mode("overwrite").save(bronze_table)
    

26/01/02 15:46:35 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


In [3]:
schema_silver = StructType([
    StructField("paciente_id", StringType(), True),
    StructField("ECG", FloatType(), True),
    StructField("RPM", FloatType(), True),
    StructField("ts", TimestampType(), True)
])

silver_table="s3a://hospital/silver/ECG_RPM"

if not DeltaTable.isDeltaTable(spark, silver_table):
    silver_df = spark.createDataFrame([], schema_silver)    
    # Guardar como tabla Delta en S3 
    silver_df.write.format("delta").mode("overwrite").save(silver_table)

In [4]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

schema_gold = StructType([
    StructField(
        "window",
        StructType([
            StructField("start", TimestampType(), True),
            StructField("end",   TimestampType(), True)
        ]),
        True  # window nullable
    ),
    StructField("paciente_id", StringType(), True),
    StructField("rpm_avg", DoubleType(), True),
    StructField("ecg_avg", DoubleType(), True),
])

gold_table="s3a://hospital/gold/ECG_RPM_minute"

if not DeltaTable.isDeltaTable(spark, gold_table):
    gold_df = spark.createDataFrame([], schema_gold)    
    # Guardar como tabla Delta en S3 
    gold_df.write.format("delta").mode("overwrite").save(gold_table)

## Capa Bronze
Objetivo: Guardar los eventos (mensajes de Kafka) tal como llegan sin validaciones (única fuente de verdad).
- Datos auditables y eprocesables
- Sin pérdida de información
- Checkpoint específico para bronze

En el stream de la capa bronze, Spark utiliza **tantas particiones como particiones tenga kafka**. Kafka se ha configurado con 3 particiones, por lo tanto Spark utilizará 3 particiones.

### Tolerancia a fallos
**failOnDataLoss = false** es una opción de tolerancia a fallos en Spark Structured Streaming con Kafka.
Cuando Spark lee de Kafka, mantiene offsets. Si Spark no puede leer un offset esperado ocurre una pérdida de datos, lo cual sucede porque:

- Kafka retiene mensajes solo X horas y se intenta acceder a mensajes que ya no existen
- El streaming estuvo detenido mucho tiempo
- Se borraron tópicos
- Hubo cambios en las particiones de Kafka
- Ocurrió una restauración desde un checkpoint viejo
  
Con **failOnDataLoss = false** Spark NO falla sino que salta los offsets perdidos y continúa procesando desde el siguiente offset disponible. Esto implica que se pierden datos, pero el stream sigue vivo. Se debe utilizar en las siguientes situaciones:

- Streaming no crítico
- Métricas agregadas (promedios, tendencias)
- Dashboards en tiempo real
- Señales biomédicas donde importa el estado actual
- Para evitar que el job se caiga en producción

Sería recomendable generar alertas para informar de dichos errores.

In [ ]:
from pyspark.sql.functions import col

raw = (spark.readStream 
    .format("kafka") 
    .option("kafka.bootstrap.servers", "kafka:9092") 
    .option("subscribe", "sensores-pacientes") 
    .option("startingOffsets", "latest") 
    .option("failOnDataLoss", "false") 
    .load()
)

# guardamos información del evento (mensaje) para su reprocesado si es necesario y su posible auditoria
bronze_df = raw.select(
    col("value").cast("string").alias("raw_value"),
    col("topic"),
    col("offset"),
    col("timestamp").alias("ingest_time"),
)

 #guarda en tabla delta el streaming
query_bronze = (bronze_df.writeStream 
     .queryName("bronze_delta_ecg_rpm") 
     .format("delta") 
     .outputMode("append") 
     .option("checkpointLocation", "s3a://hospital/bronze/ECG_RPM_checkpoint") 
     .start("s3a://hospital/bronze/ECG_RPM")
    )


## Capa Silver
Objetivo: Parsear JSON, validar rangos fisiológicos básicos y tipificar.
- Tipos correctos
- Datos confiables
- Validaciones médicas mínimas
- Checkpoint específico para silver

En esta capa como no hace operaciones tipo groupBy, window, join, o distinct, Spark sigue usando 3 particiones.

In [6]:
from pyspark.sql.functions import from_json, col
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StringType, TimestampType, IntegerType, FloatType

bronze_delta_df = (spark.readStream
        .format("delta")
        .load("s3a://hospital/bronze/ECG_RPM")
)

schema_silver = StructType([
    StructField("paciente_id", StringType(), True),
    StructField("ECG", FloatType(), True),
    StructField("RPM", FloatType(), True),
    StructField("ts", TimestampType(), True)
])

silver_df = (bronze_delta_df
        .withColumn("json", from_json(col("raw_value"), schema_silver))
        .select("json.*")
        .filter(col("paciente_id").isNotNull())
        .filter(col("ECG").between(10, 220))       # validación básica
        .filter(col("RPM").isNotNull())
    )

 #guarda en tabla delta el streaming
query_silver = (silver_df.writeStream
     .queryName("silver_delta_ecg") 
     .format("delta") 
     .outputMode("append") 
     .option("checkpointLocation", "s3a://hospital/silver/ECG_RPM_checkpoint") # checkpoint para la query de silver
     .start("s3a://hospital/silver/ECG_RPM")
    )


## Capa Gold
### Métricas por paciente cada minuto listas para análisis clínico o alertas.
- RPM y ECG promedio por paciente cada 1 minuto
- Optimizado para consumo por Dashboards, ML, ...

En los writeStream anteriores no se ha especificado .trigger(...) por lo que Spark aplica un trigger de intervalo fijo por defecto, habitualmentees del orden de 500 ms, lo que se traduce en “procesar continuamente” mientras haya datos: termina un micro‑batch y, tras un breve intervalo interno, empieza el siguiente. 

En el writeStream de gold se ha especificado **.trigger(processingTime="1 minute")** haciendo que el job dispare un micro‑batch aproximadamente cada minuto, lo que cuadra con el tamaño de ventana utilizado. Prueba la ejecución comentando esa línea. 

Pregunta ¿por qué no aparece la información al transcurrir un minuto?

### Optimización
En esta capa, dado que se utilizan operaciones window y groupBy, spark utilizará **200 particiones**, 200 tareas para un cluster de 4 cores. Esto implica un **overhead** muy alto, sobre todo porque no tenemos muchos datos, por lo que se recomienda establecer el número de particiones al doble del número de cores disponibles:

spark.conf.set("spark.sql.shuffle.partitions", 8):

Así, las agregaciones en Gold se harán en 8 particiones: 8 tareas que se ejecutan en 2 oleadas (4 + 4).

Para dejar un core para batch (recomendado) se pueden utilizar 3 cores para streaming por lo que para realizar las agregaciones en gold se deberían utilizar 6 particiones (2 por core). Esto implican 6 tareas que se realizarán en dos oleadas (3+3):

spark.conf.set("spark.sql.shuffle.partitions", 6)

- Uso óptimo de CPU
- Menos overhead
- Mejor latencia

In [7]:
from pyspark.sql.functions import window, avg, col
import time

spark.conf.set("spark.sql.shuffle.partitions", 8)

silver_delta_df = (spark.readStream
        .format("delta")   
        .load("s3a://hospital/silver/ECG_RPM")
)

gold_df = (silver_delta_df
        .withWatermark("ts", "30 seconds")
        .groupBy(
            window(col("ts"), "1 minute"),
            col("paciente_id")
        )
        .agg(
            avg("RPM").alias("rpm_avg"),
            avg("ECG").alias("ecg_avg")
        )
)

query_gold = (gold_df.writeStream
    .queryName("gold_console_ecg_rpm_minute") 
    .outputMode("append") 
    .format("delta") 
    .trigger(processingTime="60 seconds") 
    .option("checkpointLocation", "s3a://hospital/gold/ECG_RPM_minute_checkpoint") 
    .start("s3a://hospital/gold/ECG_RPM_minute"))



### Alertas de posible ataque al corazon

En esta celda se lleva a cabo al fusión de datos en batch (temperaturaSPO2_SCD1) con los datos en streaming. En este caso emitimos una alerta si:
- la temperatura es menor de 32 grados (de la tabla sp2temperatura en batch) 
- ecg es mayor de 90 (tabla streaming)


In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col
from pyspark.sql import functions as F

# este método se ejecuta con cada micro_batch procedente de la capa gold

# emitimos una alerta si:
    # la temperatura es menor de 30 grados (de la tabla sp2temperatura en batch) 
    # ecg es mayor de 90 (tabla streaming)

def process_stream_batch(micro_batch_df, batch_id):
    # Obtenemos los pacientes cuyo último estado (temperatura/SpO₂) es de las últimas 6 horas.
    # estamos asumiendo el proceso en batch se ejecuta cada 6 horas
    ahora = F.current_timestamp()
    gold_path_table_scd1 = "s3a://hospital/gold/spo2temperatura_scd1"
    scd1_df = (spark.read
        .format("delta")
        .load(gold_path_table_scd1)
        .filter(F.col("timestamp") >= ahora - F.expr("interval 6 hours"))
        .filter(F.col("temperatura") < 30)
    )
    print(f"=== BATCH {batch_id} ===")
    
    joined_stream_batch = (micro_batch_df
        .filter(F.col("ecg_avg") > 90)
        .join(
            scd1_df,
            on="paciente_id",      # o ["paciente_id"] en caso de querir añadir más claves para el join
            how="inner"            # solo pacientes con estado reciente han de tener datos de streaming y batch para hacer el join
        )
    )
   
    # solo las visualizamos pero deberíamos guardar en delta como se indica en la fila siguiente
    joined_stream_batch.show(truncate=False)
    # joined_stream_batch.write.format("delta").save("s3a://hospital/gold/alertas")
    
gold_stream_delta = (spark.readStream
    .format("delta")
    .load("s3a://hospital/gold/ECG_RPM_minute"))


query_alertas_gold=(gold_stream_delta.writeStream
    .foreachBatch(process_stream_batch)
    .option("checkpointLocation", "s3a://hospital/gold/gold_alerts_checkpoint")
    .start())

query_alertas_gold.awaitTermination()

26/01/02 15:47:47 WARN OffsetSeqMetadata: Updating the value of conf 'spark.sql.shuffle.partitions' in current session from '8' to '4'.
26/01/02 15:47:52 WARN OffsetSeqMetadata: Updating the value of conf 'spark.sql.shuffle.partitions' in current session from '8' to '4'.
26/01/02 15:47:53 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


=== BATCH 74 ===


26/01/02 15:49:22 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 60000} milliseconds, but spent 91240 milliseconds


+-----------+------------------------------------------+-----------------+-----------------+-----------+-----+--------------------------+
|paciente_id|window                                    |rpm_avg          |ecg_avg          |temperatura|SpO2 |timestamp                 |
+-----------+------------------------------------------+-----------------+-----------------+-----------+-----+--------------------------+
|4          |{2026-01-02 15:12:00, 2026-01-02 15:13:00}|28.49714265550886|96.48357173374721|25.66      |92.11|2026-01-02 12:30:00.941435|
+-----------+------------------------------------------+-----------------+-----------------+-----------+-----+--------------------------+



26/01/02 15:49:52 WARN KafkaOffsetReaderAdmin: Found incorrect offsets in some partitions (partition, previous offset, fetched offset): List((sensores-pacientes-1,1934,217), (sensores-pacientes-2,704,65))
26/01/02 15:49:52 WARN KafkaOffsetReaderAdmin: Retrying to fetch latest offsets because of incorrect offsets
26/01/02 15:49:53 WARN KafkaOffsetReaderAdmin: Found incorrect offsets in some partitions (partition, previous offset, fetched offset): List((sensores-pacientes-1,1934,218), (sensores-pacientes-2,704,65))
26/01/02 15:49:53 WARN KafkaOffsetReaderAdmin: Retrying to fetch latest offsets because of incorrect offsets
26/01/02 15:49:54 WARN KafkaOffsetReaderAdmin: Found incorrect offsets in some partitions (partition, previous offset, fetched offset): List((sensores-pacientes-1,1934,218), (sensores-pacientes-2,704,66))
26/01/02 15:50:22 WARN KafkaMicroBatchStream: Partition sensores-pacientes-1's offset was changed from 1934 to 218, some data may have been missed. 
Some data may have

=== BATCH 75 ===


... (52 outputs left out)

=== BATCH 102 ===
+-----------+------+-------+-------+-----------+----+---------+
|paciente_id|window|rpm_avg|ecg_avg|temperatura|SpO2|timestamp|
+-----------+------+-------+-------+-----------+----+---------+
+-----------+------+-------+-------+-----------+----+---------+



=== BATCH 103 ===
+-----------+------+-------+-------+-----------+----+---------+
|paciente_id|window|rpm_avg|ecg_avg|temperatura|SpO2|timestamp|
+-----------+------+-------+-------+-----------+----+---------+
+-----------+------+-------+-------+-----------+----+---------+



=== BATCH 104 ===


+-----------+------------------------------------------+-----------------+-----------------+-----------+-----+--------------------------+
|paciente_id|window                                    |rpm_avg          |ecg_avg          |temperatura|SpO2 |timestamp                 |
+-----------+------------------------------------------+-----------------+-----------------+-----------+-----+--------------------------+
|4          |{2026-01-02 17:14:00, 2026-01-02 17:15:00}|38.60523827870687|91.49476078578404|25.66      |92.11|2026-01-02 12:30:00.941435|
+-----------+------------------------------------------+-----------------+-----------------+-----------+-----+--------------------------+

